In [ ]:
import pandas as pd
import numpy as np
import gradio as gr
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from datetime import datetime, timedelta, timezone
import requests # requests 라이브러리 추가

# ==========================================
# 1. 데이터 로드
# ==========================================
# 파일 경로 (업로드된 파일명 기준)
df_calendar = pd.read_excel('academic_calendar_preprocessed.xlsx')
df_class = pd.read_excel('단과대_수업_시간_분포_전처리_단일탭.xlsx')
df_housing = pd.read_csv('단독다가구(전월세)_실거래가_20260612154920.csv', skiprows=15, encoding='euc-kr')

# ==========================================
# 2. 실시간 환경 데이터 및 분석 함수 (재사용)
# ==========================================
def get_weather_impact():
    """기상청 초단기실황 API를 이용해 실시간 날씨 정보를 가져와 가중치 및 설명 반환"""
    try:
        now = datetime.now(timezone(timedelta(hours=9))) # 한국 시간 기준

        base_date = now.strftime("%Y%m%d")
        current_hour_minute = now.hour * 100 + now.minute

        if current_hour_minute % 100 < 30:
            if now.hour == 0:
                base_time = "2330"
                base_date = (now - timedelta(days=1)).strftime("%Y%m%d")
            else:
                base_time = (now - timedelta(hours=1)).strftime("%H30")
        else:
            base_time = now.strftime("%H30")

        url = "http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst"
        params = {
            "serviceKey": "fa88db1ae3350dba4af70bf1510dda27a0edeb78da563f1eb7ef65ad457730a3", # 실제 서비스 키 사용
            "numOfRows": "10", "pageNo": "1",
            "base_date": base_date,
            "base_time": base_time,
            "nx": "73", "ny": "134", "dataType": "JSON" # 춘천시 좌표 (nx, ny)
        }

        res = requests.get(url, params=params, timeout=10)
        res.raise_for_status() # HTTP 오류가 발생하면 예외 발생
        data_json = res.json()

        items = data_json['response']['body']['items']['item']
        data = {item['category']: item['obsrValue'] for item in items}

        temp = data.get('T1H', '알 수 없음') # 기온
        pty = data.get('PTY', '0')  # 강수 형태 (0: 없음, 1: 비, 2: 비/눈, 3: 눈, 4: 소나기)
        sky = data.get('SKY', '1')  # 하늘 상태 (1: 맑음, 3: 구름많음, 4: 흐림)

        weather_impact_score = 0
        status_desc = ""

        if pty in ['1', '2', '3', '4']:
            weather_impact_score = 10
            if pty == '1': status_desc = "비"
            elif pty == '2': status_desc = "비 또는 눈"
            elif pty == '3': status_desc = "눈"
            elif pty == '4': status_desc = "소나기"
        elif sky == '3':
            status_desc = "구름많음"
        elif sky == '4':
            weather_impact_score = 5
            status_desc = "흐림"
        else:
            status_desc = "맑음"

        weather_text = f"현재 기온: {temp}℃ / 상태: {status_desc}"
        return weather_impact_score, weather_text

    except requests.exceptions.RequestException as e:
        return 0, f"날씨 데이터 요청 오류: {str(e)}"
    except KeyError as e:
        return 0, f"날씨 데이터 파싱 오류 (키 없음): {str(e)}"
    except Exception as e:
        return 0, f"날씨 데이터 수집 중 알 수 없는 오류: {str(e)}"

# 2. 데이터 학습용 변수 생성 (기존 휴리스틱을 '학습용 정답지'로 활용)
def create_model():
    data = []
    # 학사 일정 시작일과 종료일을 datetime 객체로 변환
    df_calendar['시작일'] = pd.to_datetime(df_calendar['시작일'])
    df_calendar['종료일'] = pd.to_datetime(df_calendar['종료일'])

    start_date_sim = pd.to_datetime('2026-01-01')
    end_date_sim = pd.to_datetime('2026-12-31')
    date_range_sim = pd.date_range(start=start_date_sim, end=end_date_sim)

    for _ in range(500): # 500개의 시뮬레이션 데이터 생성
        sim_date = np.random.choice(date_range_sim)

        # 시뮬레이션 날짜에 해당하는 학사 일정 영향 가중치 계산
        current_events_sim = df_calendar[(df_calendar['시작일'] <= sim_date) & (df_calendar['종료일'] >= sim_date)]
        academic_impact_sim = current_events_sim['영향_가중치'].sum() * 20 # 기존 휴리스틱과 동일한 스케일

        class_cnt_sim = np.random.randint(0, 300)
        house_cnt_sim = np.random.randint(1, 50)

        # 혼잡도 라벨 생성 (학사 일정 영향 가중치 포함)
        # 각 피처의 영향도를 고려하여 'score'를 계산하고 라벨링
        score = (academic_impact_sim * 0.1) + (class_cnt_sim * 0.01) + (house_cnt_sim * 0.05)
        label = 1 if score > 5.0 else 0 # 임계값 조정 가능
        data.append([class_cnt_sim, house_cnt_sim, academic_impact_sim, label])

    df_train = pd.DataFrame(data, columns=['class', 'house', 'academic_impact', 'label'])

    # 로지스틱 회귀 학습
    scaler = StandardScaler()
    X = scaler.fit_transform(df_train[['class', 'house', 'academic_impact']])
    y = df_train['label']

    model = LogisticRegression()
    model.fit(X, y)
    return model, scaler

model, scaler = create_model()

# [예측 함수]
def predict_congestion(college, location):
    now = datetime.now() # KST 기준은 아래 report 생성 시점에 다시 설정
    weather_impact_score, weather_desc = get_weather_impact()

    # 1) 학사 일정 (현재 날짜가 기간 내 포함되는지)
    df_calendar['시작일'] = pd.to_datetime(df_calendar['시작일'])
    df_calendar['종료일'] = pd.to_datetime(df_calendar['종료일'])
    cal_mask = (df_calendar['시작일'] <= now) & (df_calendar['종료일'] >= now)
    cal_impact_df = df_calendar.loc[cal_mask] # 현재 기간에 해당하는 학사 일정 추출
    academic_impact_feature = cal_impact_df['영향_가중치'].sum() * 20

    # 학사 일정 정보 포맷팅
    academic_events_str = ""
    if not cal_impact_df.empty:
        for idx, row in cal_impact_df.iterrows():
            academic_events_str += f"- {row['이벤트명']} ({row['시작일'].strftime('%Y-%m-%d')} ~ {row['종료일'].strftime('%Y-%m-%d')})\n"
    else:
        academic_events_str = "현재 학사 일정 중 특이사항 없음.\n"

    # 2) 수업 밀집도
    # 단과대 그룹을 분리하여 정확하게 매칭되도록 처리
    class_val = 0
    for group_str in df_class['단과대학 그룹'].unique():
        if college in [c.strip() for c in group_str.split(',')]:
            day_kr = ["월요일", "화요일", "수요일", "목요일", "금요일", "토요일", "일요일"][now.weekday()]
            class_mask = (df_class['단과대학 그룹'].str.contains(group_str)) & \
                         (df_class['요일'] == day_kr) & \
                         (df_class['시간대'] == f"{now.hour:02d}:00")
            class_val = df_class.loc[class_mask, '수업 수'].sum()
            break
    if pd.isna(class_val): class_val = 0

    # 3) 주거 밀집도
    housing_val = len(df_housing[df_housing['도로명'] == location])

    # 로지스틱 회귀 모델 입력 피처
    input_features = [[class_val, housing_val, academic_impact_feature]]
    input_scaled = scaler.transform(input_features)
    prob = model.predict_proba(input_scaled)[0][1]

    # 리포트 생성
    now_korea = datetime.now(timezone(timedelta(hours=9)))
    report = f"📅 시간: {now_korea.strftime('%Y-%m-%d %H:%M')}\n"
    report += f"☁️ 날씨: {weather_desc}\n"
    report += f"📚 현재 학사 일정:\n{academic_events_str}"
    report += f"🎯 예상 혼잡도: {prob*100:.1f}% (로지스틱 회귀 모델)\n"
    recommendation = "✅ 배출하기 좋습니다." if prob < 0.5 else "⚠️ 현재 혼잡합니다. 시간 조정을 권장합니다."

    return report, recommendation

# 단과대 선택지 다양화
all_colleges = []
for group in df_class['단과대학 그룹'].unique():
    all_colleges.extend([c.strip() for c in group.split(',')])
unique_college_choices = sorted(list(set(all_colleges)))

# [Gradio 인터페이스]
with gr.Blocks() as demo:
    gr.Markdown("# 🗑️ 춘천 대학가 스마트 쓰레기 배출 예측")

    with gr.Row():
        col_input = gr.Dropdown(choices=unique_college_choices, label="단과대")
        loc_input = gr.Dropdown(choices=df_housing['도로명'].unique().tolist(), label="거주 도로명")

    btn = gr.Button("실시간 분석")
    out_rep = gr.Textbox(label="분석 리포트")
    out_rec = gr.Textbox(label="권장 사항")

    btn.click(fn=predict_congestion, inputs=[col_input, loc_input], outputs=[out_rep, out_rec])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://15a1e0a928b43b4152.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
